In [1]:
# ===========================
#  Merge the data
# ===========================
import pandas as pd
from pathlib import Path

OUT_DIR   = Path("outputs")
ID_COL    = "essay_id_comp"
GROUP_COL = "gender"
TARGET_COL= "holistic_essay_score_3cat"

# Load test set ground truth
test_df = pd.read_csv(OUT_DIR / "test_split.csv", usecols=[ID_COL, GROUP_COL, TARGET_COL])

# Prediction files (already saved in outputs/)
pred_paths = {
    "tfidf":            OUT_DIR / "scores_tfidf_ordlogit_cv.csv",
    "bert":             OUT_DIR / "scores_bert_ord.csv",
    "chatgpt_zeroshot": OUT_DIR / "scores_chatgpt_zeroshot.csv",
    "chatgpt_fewshot":  OUT_DIR / "scores_chatgpt_fewshot.csv",
}

dfs = []
for model_key, path in pred_paths.items():
    df = pd.read_csv(path)
    pred_col = [c for c in df.columns if c.startswith("y_pred")][0]
    df = df[[ID_COL, pred_col]].rename(columns={pred_col: f"y_pred_{model_key}"})
    dfs.append(df)

# Merge all predictions onto the test set
merged = test_df.copy()
for mdf in dfs:
    merged = merged.merge(mdf, on=ID_COL, how="left")

merged_path = OUT_DIR / "scores_merged_test.csv"
merged.to_csv(merged_path, index=False)

print(">>> Merge complete (TEST only)")
print("Shape:", merged.shape)
print("Columns:", list(merged.columns))

>>> Merge complete (TEST only)
Shape: (434, 7)
Columns: ['essay_id_comp', 'gender', 'holistic_essay_score_3cat', 'y_pred_tfidf', 'y_pred_bert', 'y_pred_chatgpt_zeroshot', 'y_pred_chatgpt_fewshot']


In [2]:
merged.head()

,essay_id_comp,gender,holistic_essay_score_3cat,y_pred_tfidf,y_pred_bert,y_pred_chatgpt_zeroshot,y_pred_chatgpt_fewshot
0,7F220FD6CACF,M,1,1,2,1,1
1,B3FE48815543,F,3,3,3,2,2
2,B6428F6EB5F3,F,2,2,2,2,2
3,5D4E78CE6D5F,F,2,2,2,1,2
4,EA48C80121AD,F,2,1,1,1,1


In [3]:
# ===========================
#  Block 1 — Agreement (Kappa & QWK)
# ===========================
from sklearn.metrics import cohen_kappa_score
import pandas as pd

MODEL_COLS = {
    "tfidf": "y_pred_tfidf",
    "bert": "y_pred_bert",
    "chatgpt_zeroshot": "y_pred_chatgpt_zeroshot",
    "chatgpt_fewshot": "y_pred_chatgpt_fewshot",
}

def _pair_clean(y_true, y_pred):
    s = pd.concat([y_true, y_pred], axis=1).dropna()
    return s.iloc[:,0].astype(int), s.iloc[:,1].astype(int)

rows = []
for name, col in MODEL_COLS.items():
    yt, yp = _pair_clean(merged[TARGET_COL], merged[col])
    kappa_plain = cohen_kappa_score(yt, yp)
    kappa_quadr = cohen_kappa_score(yt, yp, weights="quadratic")
    rows.append([name, kappa_plain, kappa_quadr])

block1_df = pd.DataFrame(rows, columns=["model", "kappa", "qwk"]).set_index("model")
print("\n=== Block 1 — Agreement (vs. holistic_essay_score) ===")
print(block1_df.round(3))



=== Block 1 — Agreement (vs. holistic_essay_score) ===
                  kappa    qwk
model                         
tfidf             0.632  0.729
bert              0.731  0.799
chatgpt_zeroshot  0.254  0.400
chatgpt_fewshot   0.385  0.505


In [10]:
# ===========================
#  Block 2 — Fairness: DI (Uncond., Cond@1..3) — normalized Summary
# ===========================
import numpy as np
import pandas as pd

# Using your existing globals: merged, GROUP_COL, TARGET_COL, MODEL_COLS
true_band = merged[TARGET_COL].astype("Int64")

def _di_for_subset(pred_series, sub_idx, r):
    """
    DI for event A_r: (pred == r) within subset sub_idx.
    DI = P(A_r|F) / P(A_r|M).
    No EPS; returns NaN if undefined (e.g., no M or p_m == 0).
    """
    idx = merged.index.intersection(pd.Index(sub_idx))
    if len(idx) == 0:
        return np.nan

    idx_f = idx[merged.loc[idx, GROUP_COL] == "F"]
    idx_m = idx[merged.loc[idx, GROUP_COL] == "M"]
    n_f, n_m = len(idx_f), len(idx_m)
    if n_f == 0 or n_m == 0:
        return np.nan

    num_f = int((pred_series.loc[idx_f] == r).sum())
    num_m = int((pred_series.loc[idx_m] == r).sum())
    p_f = num_f / n_f
    p_m = num_m / n_m
    if p_m == 0:
        return np.nan
    return p_f / p_m

def _weights_for_subset(pred_series, sub_idx):
    """
    Raw class-mix weights w_r(S) = #(pred==r in S) / |S| for r=1,2,3.
    Returns dict {1: w1, 2: w2, 3: w3}. If |S|=0 -> all NaN.
    """
    idx = merged.index.intersection(pd.Index(sub_idx))
    n = len(idx)
    if n == 0:
        return {1: np.nan, 2: np.nan, 3: np.nan}
    w = {}
    for r in (1, 2, 3):
        w[r] = float((pred_series.loc[idx] == r).sum()) / n
    return w

def di_table_for_model(pred_col):
    """
    Build a 3x4 table for one model:
      Rows: Pred=1, Pred=2, Pred=3
      Cols: Uncond., Cond@1, Cond@2, Cond@3
    """
    pred = merged[pred_col].astype("Int64")

    subsets = {
        "Uncond.": merged.index,
        "Cond@1": merged.index[true_band == 1],
        "Cond@2": merged.index[true_band == 2],
        "Cond@3": merged.index[true_band == 3],
    }

    # Compute DI for each predicted class r and each subset column
    di_grid = {r: {} for r in (1, 2, 3)}
    for r in (1, 2, 3):
        for col_name, sub_idx in subsets.items():
            di_grid[r][col_name] = _di_for_subset(pred, sub_idx, r)

    # Build rows (Pred=1..3) — no Summary row
    rows = []
    for r in (1, 2, 3):
        rows.append([
            f"Pred={r}",
            di_grid[r]["Uncond."],
            di_grid[r]["Cond@1"],
            di_grid[r]["Cond@2"],
            di_grid[r]["Cond@3"],
        ])

    return (pd.DataFrame(rows, columns=["Row","Uncond.","Cond@1","Cond@2","Cond@3"])
              .set_index("Row"))


# Run for each method and print
for model_name, col in MODEL_COLS.items():
    table = di_table_for_model(col)
    print(f"\n=== Block 2 — DI (no summary) — {model_name} ===")
    print(table.round(3))



=== Block 2 — DI (no summary) — tfidf ===
        Uncond.  Cond@1  Cond@2  Cond@3
Row                                    
Pred=1    0.833   1.060   0.845     NaN
Pred=2    1.116   0.765   0.995   1.944
Pred=3    1.049     NaN   1.559   0.595

=== Block 2 — DI (no summary) — bert ===
        Uncond.  Cond@1  Cond@2  Cond@3
Row                                    
Pred=1    0.757   0.930   1.462     NaN
Pred=2    1.057   1.292   0.966   0.521
Pred=3    1.771     NaN   1.624   1.319

=== Block 2 — DI (no summary) — chatgpt_zeroshot ===
        Uncond.  Cond@1  Cond@2  Cond@3
Row                                    
Pred=1    0.815   0.904   0.863   0.556
Pred=2    1.464   3.875   1.213   1.078
Pred=3      NaN     NaN     NaN     NaN

=== Block 2 — DI (no summary) — chatgpt_fewshot ===
        Uncond.  Cond@1  Cond@2  Cond@3
Row                                    
Pred=1    0.741   0.797   0.875     NaN
Pred=2    1.295   2.510   1.064   0.965
Pred=3    1.107     NaN     NaN   0.833


In [12]:
# ===========================
#  Block 2 — Fairness: DP (Uncond., Cond@1..3) — normalized Summary
# ===========================
import numpy as np
import pandas as pd

true_band = merged[TARGET_COL].astype("Int64")

def _dp_for_subset(pred_series, sub_idx, r):
    """
    DP_r(S) = P(pred=r | F, S) - P(pred=r | M, S).
    Returns NaN only if a group is absent in S; zeros are valid results.
    """
    idx = merged.index.intersection(pd.Index(sub_idx))
    if len(idx) == 0:
        return np.nan

    idx_f = idx[merged.loc[idx, GROUP_COL] == "F"]
    idx_m = idx[merged.loc[idx, GROUP_COL] == "M"]
    n_f, n_m = len(idx_f), len(idx_m)
    if n_f == 0 or n_m == 0:
        return np.nan

    num_f = int((pred_series.loc[idx_f] == r).sum())
    num_m = int((pred_series.loc[idx_m] == r).sum())
    p_f = num_f / n_f
    p_m = num_m / n_m
    return p_f - p_m  # may be zero; keep it



def dp_table_for_model(pred_col):
    pred = merged[pred_col].astype("Int64")
    subsets = {
        "Uncond.": merged.index,
        "Cond@1": merged.index[true_band == 1],
        "Cond@2": merged.index[true_band == 2],
        "Cond@3": merged.index[true_band == 3],
    }

    # per-row values
    dp_grid = {r: {} for r in (1,2,3)}
    for r in (1,2,3):
        for col_name, sub_idx in subsets.items():
            dp_grid[r][col_name] = _dp_for_subset(pred, sub_idx, r)

    # table rows (Pred=1..3) — no "Summary" row
    rows = []
    for r in (1,2,3):
        rows.append([
            f"Pred={r}",
            dp_grid[r]["Uncond."],
            dp_grid[r]["Cond@1"],
            dp_grid[r]["Cond@2"],
            dp_grid[r]["Cond@3"],
        ])

    return (pd.DataFrame(rows, columns=["Row","Uncond.","Cond@1","Cond@2","Cond@3"])
              .set_index("Row"))

# Run for each method
for model_name, col in MODEL_COLS.items():
    table = dp_table_for_model(col)
    print(f"\n=== Block 2 — DP (no summary) — {model_name} ===")
    print(table.round(3))


=== Block 2 — DP (no summary) — tfidf ===
        Uncond.  Cond@1  Cond@2  Cond@3
Row                                    
Pred=1   -0.065   0.048  -0.020   0.000
Pred=2    0.061  -0.048  -0.004   0.283
Pred=3    0.004   0.000   0.024  -0.283

=== Block 2 — DP (no summary) — bert ===
        Uncond.  Cond@1  Cond@2  Cond@3
Row                                    
Pred=1   -0.084  -0.056   0.016   0.000
Pred=2    0.034   0.056  -0.032  -0.192
Pred=3    0.051   0.000   0.016   0.192

=== Block 2 — DP (no summary) — chatgpt_zeroshot ===
        Uncond.  Cond@1  Cond@2  Cond@3
Row                                    
Pred=1   -0.132  -0.093  -0.083  -0.067
Pred=2    0.132   0.093   0.083   0.067
Pred=3    0.000   0.000   0.000   0.000

=== Block 2 — DP (no summary) — chatgpt_fewshot ===
        Uncond.  Cond@1  Cond@2  Cond@3
Row                                    
Pred=1   -0.137  -0.179  -0.043   0.042
Pred=2    0.137   0.179   0.043  -0.033
Pred=3    0.000   0.000   0.000  -0.008
